# 01 Fetch & Chunk

AAPL/MSFT/GOOGL の 10-K × 5年 + 10-Q × 15四半期 = 60 件を取得し、
Item 1A (Risk Factors) と Item 7/Item 2 (MD&A) を抽出、
FinBERT トークナイザで 510 トークンチャンクに分割する。

In [ ]:
# Cell 1: imports + setup (必ず最初に _helpers を import)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import _helpers
_ = _helpers.setup_edgar()
device = _helpers.get_device()
print('device:', device)
print('DATA_DIR:', _helpers.DATA_DIR)


In [ ]:
# Cell 2: 10-K を 3 銘柄 × 5 年取得
from edgar.batch import BatchFetcher
from edgar.types import FilingType

TICKERS = ['AAPL', 'MSFT', 'GOOGL']
fetcher = BatchFetcher()
filings_10k = fetcher.fetch_batch(
    cik_or_tickers=TICKERS, form=FilingType.FORM_10K, limit=5, max_workers=3,
)
for t, r in filings_10k.items():
    print(t, len(r) if not isinstance(r, Exception) else f'ERR: {r}')


In [ ]:
# Cell 3: 10-Q を 3 銘柄 × 15 四半期取得
filings_10q = fetcher.fetch_batch(
    cik_or_tickers=TICKERS, form=FilingType.FORM_10Q, limit=15, max_workers=3,
)
for t, r in filings_10q.items():
    print(t, len(r) if not isinstance(r, Exception) else f'ERR: {r}')


In [ ]:
# Cell 4: filings メタを 1 つの DataFrame に統合し filings.parquet 保存
import pandas as pd

all_filing_objs = []
for d, label in [(filings_10k, '10-K'), (filings_10q, '10-Q')]:
    for ticker, result in d.items():
        if isinstance(result, Exception):
            continue
        for f in result:
            all_filing_objs.append((ticker, label, f))

df_filings = pd.DataFrame([
    {
        'filing_id': str(getattr(f, 'accession_no', '') or getattr(f, 'accession_number', '')),
        'ticker': ticker,
        'form': form,
        'filing_date': pd.Timestamp(str(getattr(f, 'filing_date', ''))),
        'accession_number': str(getattr(f, 'accession_no', '') or getattr(f, 'accession_number', '')),
    }
    for ticker, form, f in all_filing_objs
])
df_filings = df_filings.sort_values(['ticker', 'form', 'filing_date']).reset_index(drop=True)
df_filings.to_parquet(_helpers.FILINGS_PARQUET)
print('saved:', _helpers.FILINGS_PARQUET, 'rows:', len(df_filings))
df_filings.head()


In [ ]:
# Cell 5: 10-K のセクション抽出 (Item 1A, Item 7)
from edgar.cache import CacheManager
from edgar.extractors import SectionExtractor
from edgar.types import SectionKey

cache = CacheManager(cache_dir=_helpers.EDGAR_CACHE_DIR)
extractor_10k = SectionExtractor(cache=cache)

section_rows = []
for ticker, form, f in all_filing_objs:
    if form != '10-K':
        continue
    fid = str(getattr(f, 'accession_no', '') or getattr(f, 'accession_number', ''))
    for key in [SectionKey.ITEM_1A.value, SectionKey.ITEM_7.value]:
        text = extractor_10k.extract_section(f, key)
        if text:
            section_rows.append({
                'filing_id': fid, 'section_key': key,
                'text': text, 'char_count': len(text),
            })
        else:
            print(f'10-K miss: {ticker} {fid} {key}')
print('10-K sections extracted:', len(section_rows))


In [ ]:
# Cell 6: 10-Q のセクション抽出 (custom_patterns で Item 2 / Item 1A)
extractor_10q = SectionExtractor(
    cache=cache,
    custom_patterns=_helpers.SECTION_PATTERNS_10Q,
)
for ticker, form, f in all_filing_objs:
    if form != '10-Q':
        continue
    fid = str(getattr(f, 'accession_no', '') or getattr(f, 'accession_number', ''))
    for key in ['item_7', 'item_1a']:
        text = extractor_10q.extract_section(f, key)
        if text:
            section_rows.append({
                'filing_id': fid, 'section_key': key,
                'text': text, 'char_count': len(text),
            })
        else:
            print(f'10-Q miss: {ticker} {fid} {key}')
print('total sections:', len(section_rows))


In [ ]:
# Cell 7: sections.parquet 保存
df_sections = pd.DataFrame(section_rows)
df_sections.to_parquet(_helpers.SECTIONS_PARQUET)
print('saved:', _helpers.SECTIONS_PARQUET, 'rows:', len(df_sections))
df_sections.groupby(['section_key']).size()


In [ ]:
# Cell 8: FinBERT トークナイザでチャンク化
from tqdm.auto import tqdm
tokenizer, _model = _helpers.load_finbert()
del _model  # チャンク化はトークナイザのみ必要

chunk_rows = []
for row in tqdm(df_sections.to_dict('records'), desc='chunking'):
    chunks = _helpers.chunk_text(row['text'], tokenizer)
    for i, c in enumerate(chunks):
        chunk_rows.append({
            'filing_id': row['filing_id'],
            'section_key': row['section_key'],
            'chunk_idx': i,
            'text': c,
            'token_count': len(tokenizer.encode(c, add_special_tokens=False)),
        })
print('total chunks:', len(chunk_rows))


In [ ]:
# Cell 9: chunks.parquet 保存 + ticker/form 結合
df_chunks = pd.DataFrame(chunk_rows).merge(
    df_filings[['filing_id', 'ticker', 'form', 'filing_date']],
    on='filing_id', how='left',
)
df_chunks.to_parquet(_helpers.CHUNKS_PARQUET)
print('saved:', _helpers.CHUNKS_PARQUET, 'rows:', len(df_chunks))
df_chunks.groupby(['ticker', 'form', 'section_key']).size()
